# Day 18: SQL 窗口函数 —— CTE + ROW_NUMBER/RANK/LEAD/LAG 习题

> **范围**: CTE 临时命名子查询 + ROW_NUMBER/RANK/DENSE_RANK + LEAD/LAG + SUM OVER
> **数据**: `../data/sales.csv` + `../data/customers.csv`
> **建议用时**: 60-90 分钟
> **注意**: CTE 用 `WITH name AS (SELECT ...)`，窗口函数用 `OVER (PARTITION BY ... ORDER BY ...)`

In [2]:
import duckdb 

%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

## Easy

**1. CTE 基础 —— 清洗后分组**

基于 `../data/sales.csv`：
- 写 CTE `clean_sales`：把 `product` 转为小写并去空格，用 `TRIM(LOWER(product))`
- 从 CTE 中按 `product_clean` 分组，统计订单数和总销售额
- 按总销售额降序，显示前10

In [4]:
%%sql
WITH clean_sales AS (
    SELECT
    TRIM(LOWER(product)) AS product_clean,
    total
    FROM '../data/sales.csv'
)
SELECT
    product_clean,
    COUNT(*) AS num_sales,
    SUM(total) AS total_sales
FROM clean_sales
GROUP BY product_clean
ORDER BY total_sales DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

product_clean,num_sales,total_sales
phone,93,257831
keyboard,99,225111
mouse,81,216155
headphones,67,204686
laptop,87,200939
monitor,73,162994


**2. CTE 多步骤 —— 两个 CTE 串联**

基于 `../data/sales.csv`：
- CTE `sales_base`：选择 `order_id`, `customer_id`, `total`, `country`
- CTE `country_stats`：从 `sales_base` 按 `country` 分组，求 `COUNT(*)` 和 `SUM(total)`
- 主查询：从 `country_stats` 筛选总销售额 > 20000 的国家，按销售额降序

In [5]:
%%sql
WITH sales_base AS(
    SELECT 
    order_id,
    customer_id,
    total,
    country
    FROM '../data/sales.csv'
),
country_stats AS (
    SELECT
    country,
    COUNT(*) AS num_orders,
    SUM(total) AS total_sales
    FROM sales_base
    GROUP BY country
)
SELECT * FROM country_stats
WHERE total_sales > 20000
ORDER BY total_sales DESC;

Running query in 'duckdb:///:memory:'

country,num_orders,total_sales
UK,197,534817
US,125,300613
France,80,208165
Germany,66,144020
China,32,80101


**3. ROW_NUMBER —— 组内排名**

基于 `../data/sales.csv`：
- 用 `ROW_NUMBER() OVER (PARTITION BY country ORDER BY total DESC)` 给每个国家的订单按金额排名
- 显示 `order_id`, `country`, `total`, `rn`（排名）
- 用 CTE 筛选出每个国家的 TOP 1 订单（`rn = 1`）
- 显示结果

In [7]:
%%sql
WITH ranked AS (
    SELECT
    order_id,
    country,
    total,
    ROW_NUMBER() OVER (PARTITION BY country ORDER BY total DESC) AS rn
    FROM '../data/sales.csv'
)
SELECT * FROM ranked
WHERE rn = 1
ORDER BY country;


Running query in 'duckdb:///:memory:'

order_id,country,total,rn
O1035,China,6495,1
O1319,France,9995,1
O1342,Germany,9995,1
O1033,UK,9995,1
O1238,US,9995,1


## Medium

**4. RANK vs DENSE_RANK —— 对比并列**

基于 `../data/sales.csv`：
- 筛选 `country = 'US'` 的订单
- 用 `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()` 三种方式按 `total` 降序排名
- 显示 `order_id`, `total`, `rn`, `rnk`, `drnk`
- 观察：当有两条订单的 `total` 相同时，三种排名的区别是什么？

In [8]:
%%sql
SELECT 
order_id, total, 
ROW_NUMBER() OVER (PARTITION BY country ORDER BY total DESC) AS rn,
RANK() OVER (PARTITION BY country ORDER BY total DESC) AS rnk,
DENSE_RANK() OVER (PARTITION BY country ORDER BY total DESC) AS drnk
FROM '../data/sales.csv'
WHERE country = 'US'
ORDER BY total DESC
LIMIT 10;


Running query in 'duckdb:///:memory:'

order_id,total,rn,rnk,drnk
O1238,9995,1,1,1
O1470,9995,2,1,1
O1432,9995,3,1,1
O1117,7996,4,4,2
O1092,7996,5,4,2
O1083,6495,6,6,3
O1218,6495,7,6,3
O1160,6495,8,6,3
O1234,6495,9,6,3
O1390,5997,10,10,4


**5. LEAD / LAG —— 环比计算**

基于 `../data/sales.csv`：
- 用 CTE `monthly` 按 `STRFTIME('%Y-%m', order_date)` 分组，统计每月总销售额
- 在主查询中用 `LAG(total_sales, 1) OVER (ORDER BY ym)` 取上月销售额
- 计算每月环比变化额（`total_sales - LAG(...)`）
- 计算每月环比变化率（百分比，保留1位小数）
- 注意：第一行的 `LAG` 为 NULL，用 `COALESCE(LAG(...), 0)` 处理

In [ ]:
%%sql
WITH monthly AS (
    SELECT
    STRFTIME('%Y-%m', order_date) AS ym,
    SUM(total) AS total_sales
    FROM '../data/sales.csv'
    GROUP BY ym
    ORDER BY ym
)
SELECT
  ym,
  total_sales,
  COALESCE(LAG(total_sales, 1) OVER (ORDER BY ym), 0) AS prev_month,
  total_sales - LAG(total_sales, 1) OVER (ORDER BY ym) AS change,
  ROUND(
    (total_sales - LAG(total_sales, 1) OVER (ORDER BY ym)) * 100.0 / LAG(total_sales, 1) OVER (ORDER BY ym),
    1
  ) AS change_pct
FROM monthly;

Running query in 'duckdb:///:memory:'

ym,total_sales,prev_month,change,change_pct
2024-01,99365,0,None,None
2024-02,81795,99365,-17570,-17.7
2024-03,133371,81795,51576,63.1
2024-04,123882,133371,-9489,-7.1
2024-05,71176,123882,-52706,-42.5
2024-06,119061,71176,47885,67.3
2024-07,101271,119061,-17790,-14.9
2024-08,104388,101271,3117,3.1
2024-09,103183,104388,-1205,-1.2
2024-10,117265,103183,14082,13.6


参考答案

你的问题：COALESCE 只处理了 prev_month 展示列，但 change 列仍用原始 LAG 计算，导致第一行为 NULL。

修正核心：COALESCE 要用于计算表达式，不只是展示列。应该把 COALESCE 的结果存为 CTE 列，再用该列计算 change。

In [27]:
%%sql
WITH monthly AS (
    SELECT
        STRFTIME('%Y-%m', order_date) AS ym,
        SUM(total) AS total_sales
    FROM '../data/sales.csv'
    GROUP BY ym
    ORDER BY ym
),
monthly_with_prev AS (
    SELECT
        ym,
        total_sales,
        COALESCE(LAG(total_sales, 1) OVER (ORDER BY ym), 0) AS prev_month
    FROM monthly
)
SELECT
    ym,
    total_sales,
    prev_month,
    total_sales - prev_month AS change,
    CASE 
        WHEN prev_month = 0 THEN NULL
        ELSE ROUND((total_sales - prev_month) * 100.0 / prev_month, 1)
    END AS change_pct
FROM monthly_with_prev;

Running query in 'duckdb:///:memory:'

ym,total_sales,prev_month,change,change_pct
2024-01,99365,0,99365,None
2024-02,81795,99365,-17570,-17.7
2024-03,133371,81795,51576,63.1
2024-04,123882,133371,-9489,-7.1
2024-05,71176,123882,-52706,-42.5
2024-06,119061,71176,47885,67.3
2024-07,101271,119061,-17790,-14.9
2024-08,104388,101271,3117,3.1
2024-09,103183,104388,-1205,-1.2
2024-10,117265,103183,14082,13.6


**6. SUM OVER —— 累计占比**

基于 `../data/sales.csv`：
- 按 `country` 分组，计算每个国家的总销售额（用 `SUM(total) OVER (PARTITION BY country)`）
- 计算每个订单占其国家总销售额的百分比（保留1位小数）
- 用 CTE 筛选出占比 > 10% 的订单（大单），显示 `order_id`, `country`, `total`, `pct`
- 按国家排序

In [18]:
%%sql
WITH orders AS(
    SELECT
    order_id,
    country,
    SUM(total) OVER (PARTITION BY country) AS country_total,
    ROUND(total * 100.0 / SUM(total) OVER (PARTITION BY country), 1) AS pct
    FROM '../data/sales.csv'
)
SELECT 
order_id,
country,
country_total,
pct
FROM orders
WHERE pct > 10
ORDER BY country


Running query in 'duckdb:///:memory:'

order_id,country,country_total,pct


**7. CTE + JOIN + 窗口函数综合**

基于 `../data/sales.csv` 和 `../data/customers.csv`：
- CTE `sales_with_cust`：LEFT JOIN `sales` 和 `customers`，保留 `order_id`, `total`, `name`, `country`
- 对 `sales_with_cust` 用 `RANK() OVER (PARTITION BY country ORDER BY total DESC)` 给每个国家的客户排名
- 用 CTE 筛选每个国家排名 <= 3 的客户
- 显示 `country`, `name`, `total`, `rnk`

In [20]:
%%sql
WITH sales_with_cust AS (
    SELECT
        s.order_id,
        s.total,
        c.name,
        c.country
    FROM '../data/sales.csv' s
    LEFT JOIN '../data/customers.csv' c
        ON s.customer_id = c.customer_id
),
country_rank AS (
    SELECT
        country,
        name,
        total,
        RANK() OVER (PARTITION BY country ORDER BY total DESC) AS rnk
    FROM sales_with_cust
)
SELECT country, name, total, rnk
FROM country_rank
WHERE rnk <= 3
ORDER BY country, rnk;


Running query in 'duckdb:///:memory:'

country,name,total,rnk
China,Frank,9995,1
China,Frank,9995,1
China,Frank,9995,1
China,Frank,9995,1
China,Frank,9995,1
France,Charlie,9995,1
France,Alice,9995,1
France,Alice,9995,1
France,Charlie,9995,1
Germany,Grace,9995,1


参考答案

你的问题：RANK 是在订单粒度排名，不是客户粒度。同一个客户的多笔订单都出现在结果中，导致同一个客户重复。

修正核心：先 GROUP BY customer_id 聚合每个客户的总消费，再 RANK。

In [28]:
%%sql
WITH sales_with_cust AS (
    SELECT
        s.order_id,
        s.customer_id,
        s.total,
        c.name,
        c.country
    FROM '../data/sales.csv' s
    LEFT JOIN '../data/customers.csv' c
        ON s.customer_id = c.customer_id
),
customer_total AS (
    -- 先按客户聚合，把粒度从「订单」升到「客户」
    SELECT
        country,
        name,
        SUM(total) AS customer_total
    FROM sales_with_cust
    GROUP BY country, name
),
country_rank AS (
    SELECT
        country,
        name,
        customer_total,
        RANK() OVER (PARTITION BY country ORDER BY customer_total DESC) AS rnk
    FROM customer_total
)
SELECT country, name, customer_total, rnk
FROM country_rank
WHERE rnk <= 3
ORDER BY country, rnk;

Running query in 'duckdb:///:memory:'

country,name,customer_total,rnk
China,Frank,198806,1
France,Alice,184001,1
France,Charlie,160212,2
Germany,Grace,143917,1
Germany,Bob,140967,2
UK,Henry,164484,1
US,David,144095,1
US,Eva,131234,2


## Hard

**8. 累计求和 ——  running total**

基于 `../data/sales.csv`：
- CTE `daily`：按 `order_date` 分组，统计每天的总销售额（`SUM(total)`）
- 在主查询中计算「累计销售额」：`SUM(total_sales) OVER (ORDER BY order_date)`
  （这是窗口函数不带 `PARTITION BY` 的用法，整个结果集就是一个窗口）
- 显示 `order_date`, `daily_sales`, `running_total`
- 找出累计销售额首次突破 100000 的日期

In [23]:
%%sql
WITH daily AS (
    SELECT
        order_date,
        SUM(total) AS daily_sales
    FROM '../data/sales.csv'
    GROUP BY order_date
),
daily_cum AS (
    SELECT
        order_date,
        daily_sales,
        SUM(daily_sales) OVER (ORDER BY order_date) AS running_total
    FROM daily
)
SELECT *
FROM daily_cum
WHERE running_total >= 100000
ORDER BY order_date
LIMIT 1;

Running query in 'duckdb:///:memory:'

order_date,daily_sales,running_total
2024-02-01,2997,102362


**9. 同比分析 —— LEAD/LAG 进阶**

基于 `../data/sales.csv`：
- CTE `monthly`：按年月分组，统计每月总销售额
- 在主查询中用 `LAG(total_sales, 12) OVER (ORDER BY ym)` 取「去年同期」销售额
  （假设数据跨越至少两年，LAG 12 就是去年同月）
- 计算同比变化额和同比变化率
- 用 `COALESCE` 处理 NULL（没有去年同期的月份显示为 NULL）
- 筛选出有同比数据的月份（即去年同期不为 NULL）

In [25]:
%%sql
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', order_date) AS ym,
        SUM(total) AS total_sales
    FROM '../data/sales.csv'
    GROUP BY ym
),
monthly_yoy AS (
    SELECT
        ym,
        total_sales,
        LAG(total_sales, 12) OVER (ORDER BY ym) AS last_year_sales,
        COALESCE(total_sales - LAG(total_sales, 12) OVER (ORDER BY ym), NULL) AS yoy_diff,
        COALESCE(
            ROUND(
                (total_sales - LAG(total_sales, 12) OVER (ORDER BY ym)) * 1.0
                / NULLIF(LAG(total_sales, 12) OVER (ORDER BY ym), 0),
                4
            ),
            NULL
        ) AS yoy_rate
    FROM monthly
)
SELECT *
FROM monthly_yoy
WHERE last_year_sales IS NOT NULL
ORDER BY ym;

Running query in 'duckdb:///:memory:'

ym,total_sales,last_year_sales,yoy_diff,yoy_rate


**10. 综合管道 —— 客户 RFM 分析（SQL 版）**

基于 `../data/sales.csv`：

**阶段1 —— CTE 计算 RFM 指标**:
```sql
WITH rfm_base AS (
  SELECT
    customer_id,
    MAX(order_date) AS last_order_date,
    COUNT(*) AS frequency,
    SUM(total) AS monetary
  FROM '../data/sales.csv'
  GROUP BY customer_id
)
```

**阶段2 —— 计算 R 值（距今天数）**:
- 假设 today = `2024-06-01`
- `DATE_DIFF('day', last_order_date, CAST('2024-06-01' AS DATE))`

**阶段3 —— 用 NTILE 分箱（窗口函数）**:
- `NTILE(3) OVER (ORDER BY recency DESC)` —— R 越小越好，所以 DESC
- `NTILE(3) OVER (ORDER BY frequency ASC)` —— F 越大越好，所以 ASC
- `NTILE(3) OVER (ORDER BY monetary ASC)` —— M 越大越好，所以 ASC

**阶段4 —— 输出**:
- 显示 `customer_id`, `recency`, `frequency`, `monetary`, `r_score`, `f_score`, `m_score`
- 按 `monetary` 降序，显示前10名

In [26]:
%%sql
WITH rfm_base AS (
  SELECT
    customer_id,
    MAX(order_date) AS last_order_date,
    COUNT(*) AS frequency,
    SUM(total) AS monetary
  FROM '../data/sales.csv'
  GROUP BY customer_id
),
rfm_recency AS (
  SELECT
    *,
    DATE_DIFF('day', last_order_date, CAST('2024-06-01' AS DATE)) AS recency
  FROM rfm_base
),
rfm_score AS (
  SELECT
    customer_id,
    recency,
    frequency,
    monetary,
    NTILE(3) OVER (ORDER BY recency DESC) AS r_score,
    NTILE(3) OVER (ORDER BY frequency ASC) AS f_score,
    NTILE(3) OVER (ORDER BY monetary ASC) AS m_score
  FROM rfm_recency
)
SELECT
  customer_id,
  recency,
  frequency,
  monetary,
  r_score,
  f_score,
  m_score
FROM rfm_score
ORDER BY monetary DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

customer_id,recency,frequency,monetary,r_score,f_score,m_score
C006,-208,62,198806,1,2,3
C001,-212,71,184001,3,2,3
C008,-206,72,164484,1,3,2
C003,-210,64,160212,2,2,2
C004,-213,76,144095,3,3,2
C007,-210,58,143917,2,1,1
C002,-184,46,140967,1,1,1
C005,-209,51,131234,2,1,1


参考答案

你的问题：today = 2024-06-01 在订单日期之前，导致 recency 为负数。NTILE 对负数分箱逻辑混乱。

修正核心：先确认数据时间范围，用 MAX(order_date) 或选一个在所有日期之后的今天。

In [29]:
%%sql
-- 先确认数据时间范围
WITH date_check AS (
    SELECT MAX(order_date) AS max_date FROM '../data/sales.csv'
),
-- 阶段1: 计算 RFM 指标
rfm_base AS (
  SELECT
    customer_id,
    MAX(order_date) AS last_order_date,
    COUNT(*) AS frequency,
    SUM(total) AS monetary
  FROM '../data/sales.csv'
  GROUP BY customer_id
),
-- 阶段2: 计算 R 值（用数据最大日期+1天作为 today，避免负数）
rfm_recency AS (
  SELECT
    r.*,
    DATE_DIFF('day', last_order_date, 
        CAST((SELECT max_date FROM date_check) AS DATE) + INTERVAL '1' DAY
    ) AS recency
  FROM rfm_base r
),
-- 阶段3: NTILE 分箱
rfm_score AS (
  SELECT
    customer_id,
    recency,
    frequency,
    monetary,
    NTILE(3) OVER (ORDER BY recency DESC) AS r_score,
    NTILE(3) OVER (ORDER BY frequency ASC) AS f_score,
    NTILE(3) OVER (ORDER BY monetary ASC) AS m_score
  FROM rfm_recency
)
SELECT
  customer_id,
  recency,
  frequency,
  monetary,
  r_score,
  f_score,
  m_score
FROM rfm_score
ORDER BY monetary DESC
LIMIT 10;

Running query in 'duckdb:///:memory:'

customer_id,recency,frequency,monetary,r_score,f_score,m_score
C006,6,62,198806,1,2,3
C001,2,71,184001,3,2,3
C008,8,72,164484,1,3,2
C003,4,64,160212,2,2,2
C004,1,76,144095,3,3,2
C007,4,58,143917,2,1,1
C002,30,46,140967,1,1,1
C005,5,51,131234,2,1,1
